In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re
import scipy
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
df = pd.read_csv("/content/drive/MyDrive/data_goodreads.csv")
df = df.head(300000)

In [4]:
def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return None

def parse_genres(x):
    if not isinstance(x, str):
        return []
    return re.findall(r'"(.*?)"', x)

def encode_genres(genres):
  vec = np.zeros(30, dtype=np.float32)
  for g in genres:
      if g in genre_to_idx:
          vec[genre_to_idx[g]] = 1.0
  return vec

bad_rows = df[df["genres"].apply(lambda x: safe_eval(x) is None)]
df["genres"] = df["genres"].apply(parse_genres)
top30 = df["genres"].explode().value_counts().head(30)
top30 = df["genres"].explode().value_counts().head(30).index.tolist()
genre_to_idx = {g: i for i, g in enumerate(top30)}
y = np.stack(df["genres"].apply(encode_genres))

# print(top30)
# print(bad_rows)

In [5]:
class MLP_basic(nn.Module):
    def __init__(self, input_dim=5000, num_classes=30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class MLP_w2v(nn.Module):
    def __init__(self, input_dim=300, num_classes=30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
          )

    def forward(self, x):
        return self.net(x)

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)  # (B, 30)
        loss = criterion(logits, y.float())

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

    return total_loss / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.2):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs > threshold).cpu().numpy()

        all_preds.append(preds)
        all_targets.append(y.cpu().numpy())

        total_loss += loss.item() * x.size(0)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    micro_f1 = f1_score(all_targets, all_preds, average="micro")

    return total_loss / len(loader.dataset), micro_f1

In [7]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SparseTextDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x_row = self.X[idx].toarray().flatten()
        return torch.tensor(x_row, dtype=torch.float32), self.y[idx]

In [27]:
def load_X(embedding_method= 'w2v'):
  if embedding_method == 'w2v':
    container = np.load('/content/drive/MyDrive/data/w2v_emb.npz')
    X = container['w2v_vectors']
    X = X[:300000]

  if embedding_method == 'tfidf':
    X = scipy.sparse.load_npz('/content/drive/MyDrive/data/tfidf_emb.npz')
    X = X[:300000]

  if embedding_method == 'bow':
    X = scipy.sparse.load_npz('/content/drive/MyDrive/data/bow_emb.npz')
    X = X[:300000]

  return X




def create_loader(X, y, embedding_method= 'w2v'):

  if embedding_method == 'w2v':
    dataset = TextDataset(X, y)

  else:
    dataset = SparseTextDataset(X, y)

  loader = DataLoader(dataset, batch_size=32, shuffle=True)

  return loader

# funkcja ucząca

def experiment(X, y, embedding_method = 'w2v'):

  device = "cuda" if torch.cuda.is_available() else "cpu"

  if embedding_method == 'w2v':
    model = MLP_w2v().to(device)

  else:
    model = MLP_basic().to(device)

  optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

  pos_counts = y.sum(axis=0)
  neg_counts = len(y) - pos_counts

  pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)

  criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

  #criterion = nn.BCEWithLogitsLoss()

  loader = create_loader(X, y, embedding_method=embedding_method)

  for epoch in range(3):
      train_loss = train_one_epoch(model, loader, optimizer, criterion, device)
      val_loss, val_f1 = evaluate(model, loader, criterion, device)

      print(f"Epoch {epoch}")
      print(f"Train loss: {train_loss:.4f}")
      print(f"Val loss:   {val_loss:.4f}")
      print(f"Val micro-F1: {val_f1:.4f}")

In [31]:
# w2v
m = 'w2v'
X = load_X(embedding_method= m)
experiment(X, y, embedding_method= m)

Epoch 0
Train loss: 0.7777
Val loss:   0.7051
Val micro-F1: 0.2242
Epoch 1
Train loss: 0.6956
Val loss:   0.6691
Val micro-F1: 0.2295
Epoch 2
Train loss: 0.6706
Val loss:   0.6480
Val micro-F1: 0.2440


In [28]:
# bow
m = 'tfidf'
X = load_X(embedding_method=m)
experiment(X, y, embedding_method=m)

Epoch 0
Train loss: 0.7006
Val loss:   0.5600
Val micro-F1: 0.2460
Epoch 1
Train loss: 0.5568
Val loss:   0.4493
Val micro-F1: 0.2834
Epoch 2
Train loss: 0.4312
Val loss:   0.3294
Val micro-F1: 0.3609


In [29]:
# bow
m = 'bow'
X = load_X(embedding_method=m)
experiment(X, y, embedding_method=m)

Epoch 0
Train loss: 0.7084
Val loss:   0.5437
Val micro-F1: 0.2430
Epoch 1
Train loss: 0.5414
Val loss:   0.4241
Val micro-F1: 0.2934
Epoch 2
Train loss: 0.4150
Val loss:   0.3273
Val micro-F1: 0.3645


Dla 3 epoch, bow i tfidf wykonywaly sie 2 razy dłużej od w2v, a sieć nadal jest bardzo mała. Co będzie później???